# Running cellpose with GPUs

In [1]:
import numpy as np
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

from skimage import io
from skimage.filters import threshold_otsu

import zarr
from cellpose import models, core

use_GPU = core.use_gpu(gpu_number=0)
print('>>> GPU activated? %d'%use_GPU)

import ray

from dask import array as da

import mFISHwarp.morphology
import mFISHwarp.utils
import mFISHwarp.zarr

import pandas as pd

2024-08-20 14:00:49,454	INFO util.py:154 -- Outdated packages:
  ipywidgets==7.6.5 found, needs ipywidgets>=8
Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


>>> GPU activated? 1


## Load model

In [2]:
# path to dataset and model
dataset_folder = "/mnt/ampa02_data01/tmurakami/model_training/crops_tophat"

train_folder = os.path.join(dataset_folder,'training')
models_path = os.path.join(train_folder,'models')

models_file = os.listdir(models_path); models_file.sort()
model_path = os.path.join(train_folder,'models',models_file[-1])

model = models.CellposeModel(gpu=use_GPU, pretrained_model=model_path)

## Load N5 or Zarr

In [3]:
# data_path = '/mnt/ampa02_data01/tmurakami/240417_whole_4color_1st_M037-3pb/registration/round02.zarr'
data_path = '/mnt/ampa02_data01/tmurakami/240417_whole_4color_1st_M037-3pb/fused/fused.n5'

res_analysis = 0 # resolution to be analyzed. usually the highest resolution.
res_mask = 4 # resolution to make mask. this does not need to be high.

# lazily load images using dask
imgs = mFISHwarp.zarr.omezarr_bdn5_to_dask(data_path,resolution=res_analysis)
imgs_mask = mFISHwarp.zarr.omezarr_bdn5_to_dask(data_path,resolution=res_mask)

In [4]:
# decide which channel to analyze
segment_chan = 1
reference_chan = 3
name_segment_chan = 'Test'
segment_save_dir = None

if segment_save_dir is None:
    segment_save_dir = os.path.join(os.path.dirname(os.path.dirname(data_path)),'segmentation')

## Create mask

In [5]:
# downsampling
img_down_ref = imgs_mask[reference_chan,...].compute()
global_thresh = threshold_otsu(img_down_ref)
img_mask = mFISHwarp.morphology.mask_maker(img_down_ref,global_thresh)

# import napari
# viewer = napari.Viewer()
# viewer.add_image(img_mask)
# viewer.add_image(img_down_ref)

## Make overlapped images

In [6]:
### Parameters
auto_diam = False # Cellpose automatic diameter estimation.
# theoretically, anisotropy parameter affects the accuracy. However in practice, changing this values to be the exact voxel ratio does not significantly add accuracy. 
# this may be because of the non-isotropic PSF of light-sheet.
voxel_size = (2.0,1.3,1.3)
anisotropy = voxel_size[1]/voxel_size[0]
min_size = 40

# Channel parameters which were used during the training.
Training_channel = 2 # I do not know but the cellpose see the images as KRGB. If the color is green, set it to 2.
Second_training_channel = 1

# lazyly read image and convert to dask array
chunk_size = (256,512,512)
depth = (32,64,64) 
boundary = "reflect"

### Make overlapping images
# make overlapped images for both refernce and target
overlap_imgs = []

# reference
img_ref = imgs[reference_chan,...]
img_ref = da.rechunk(img_ref,chunks=chunk_size)
overlap_imgs.append(da.overlap.overlap(img_ref, depth, boundary))
# target
img = imgs[segment_chan,...]
img = da.rechunk(img,chunks=chunk_size)
overlap_imgs.append(da.overlap.overlap(img, depth, boundary))

# If mask is used, calculate which chunks will be segmented
flag_array = mFISHwarp.utils.flag_array_generator(chunk_size, img_ref.shape, img_mask)
print(f'{flag_array.sum()} blocks of {flag_array.shape[0]}*{flag_array.shape[1]}*{flag_array.shape[2]}={flag_array.size} blocks will be calculated')

1410 blocks of 10*17*13=2210 blocks will be calculated


## Prepare zarr container to save segmentation

In [7]:
labeled_overlap_zarr_path = os.path.join(segment_save_dir,name_segment_chan,'segmented_overlap.zarr')
labeled_overlap_zarr = zarr.open(
    labeled_overlap_zarr_path,
    mode='a', 
    shape=overlap_imgs[0].shape, 
    chunks=mFISHwarp.utils.chunks_from_dask(overlap_imgs[0]), 
    dtype=np.int32)

# labeled_overlap_zarr = zarr.open(labeled_overlap_zarr_path,mode='a')

prob_zarr_path = os.path.join(segment_save_dir,name_segment_chan,'prob.zarr')
# prob_overlap_zarr = zarr.open(
#     prob_overlap_zarr_path,
#     mode='a', 
#     shape=overlap_imgs[0].shape, 
#     chunks=mFISHwarp.utils.chunks_from_dask(overlap_imgs[0]), 
#     dtype=np.float16)

prob_zarr = zarr.open(
    prob_zarr_path,
    mode='a', 
    shape=img.shape, 
    chunks=mFISHwarp.utils.chunks_from_dask(img), 
    dtype=np.float16)


# add attribute so that I can know the overlap size later.
labeled_overlap_zarr.attrs.update({"overlap_size": depth})
prob_zarr.attrs.update({"overlap_size": depth})

In [8]:
from cupyx.scipy.ndimage import white_tophat
from skimage.morphology import ball
import cupy as cp

def preprocessing(img, ball_size, norm_values_ref, norm_values_tar):
    """
    Define your preprocessing here. 
    In this specific case, I used tophat fileter and normalization.
    img: (c,z,y,x)
    """
    
    # tophat filter
    img = img.astype(float)
    footprint = ball(ball_size)
    footprint_cu = cp.asarray(footprint)
    res = []
    for i in range(img.shape[0]):
        img_cp = cp.asarray(img[i,...])
        res.append(cp.asnumpy(white_tophat(img_cp,footprint=footprint_cu)))
        del img_cp
    
    del footprint_cu
    cp._default_memory_pool.free_all_blocks()

    # normalization
    img = np.stack([
        mFISHwarp.utils.normalization_two_values(res[0], norm_values_ref[0], norm_values_ref[1]),
        mFISHwarp.utils.normalization_two_values(res[1], norm_values_tar[0], norm_values_tar[1])
    ])
    
    
    return img

In [9]:
@ray.remote(num_gpus=0.5,max_calls=1)
def segmentor(
    chunks, # list of images. reference and target
    channels,
    model,
    anisotropy,
    index,
    min_size,
    zarr_file,
    prob_zarr_file,
    *args,
    **kwargs
):
    # convert dask array to numpy array.
    chunks = np.stack([i.compute() for i in chunks])
    
    # do your defined preprocessing if necessary.
    chunks = preprocessing(chunks,*args,**kwargs)
    
    # run cellpose
    segments, flow, _  = model.eval(chunks, channels=channels, normalize=False, z_axis=1, diameter=model.diam_mean, do_3D=True, min_size=min_size, progress=False, anisotropy=anisotropy, tile=False)
    segments = segments.astype(np.int32)
    prob = flow[2].astype(np.float16) # float 16 to save data size
    
    # save masks to zarr
    chunk_info = da.from_zarr(zarr_file).chunks
    zarr_file[mFISHwarp.utils.obtain_chunk_slicer(chunk_info, index)] = segments
    
    # save probability to zarr
    if prob_zarr_file is not None:
        prob_chunk_info = da.from_zarr(prob_zarr_file).chunks
        overlap_size = prob_zarr_file.attrs['overlap_size']
        shape = [i[j] for i, j in zip(prob_chunk_info, index)]
        slicer = tuple(slice(i, i + j) for i, j in zip(overlap_size, shape))
        prob_zarr_file[mFISHwarp.utils.obtain_chunk_slicer(prob_chunk_info, index)] = prob[slicer]

In [ ]:
### in case from the middle of the computation
stored_chunks = os.listdir(labeled_overlap_zarr_path)
stored_chunks.sort()

idxs = mFISHwarp.utils.get_dask_index(overlap_imgs[0])

diameter_yx = model.diam_mean
anisotropy = anisotropy

min_size = 40
model_type = model
channels = [Training_channel, Second_training_channel]

# parameters for pre-processing
ball_size = 10
normalization_metadata = "/mnt/ampa02_data01/tmurakami/model_training/norm_values_tophat.pickle"
data_path = '/mnt/ampa02_data01/tmurakami/240417_whole_4color_1st_M037-3pb/fused/top_hat_10.zarr'
norm_info = pd.read_pickle(normalization_metadata)
norm_values_ref = [norm_info[data_path][reference_chan]['lower'], norm_info[data_path][reference_chan]['upper']]
norm_values_tar = [norm_info[data_path][segment_chan]['lower'], norm_info[data_path][segment_chan]['upper']]

for index in idxs:
    if flag_array[index[0],index[1],index[2]]:
        if '.'.join([str(i) for i in index]) not in stored_chunks:
            input_blocks = [mFISHwarp.utils.slicing_with_chunkidx(img, index) for img in overlap_imgs]
            segmentor.remote(
                input_blocks,
                [Training_channel, Second_training_channel],
                model,
                anisotropy,
                index,
                min_size,
                labeled_overlap_zarr,
                prob_zarr,
                ball_size,
                norm_values_ref,
                norm_values_tar
            )

2024-08-20 14:01:02,555	INFO worker.py:1743 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
